# Phase 2: Load and profile Liverpool City Council spend (Jan–Jun 2025)

## 1. Load all six CSV files into DuckDB

In [1]:
import duckdb
con = duckdb.connect("../council.duckdb")
con.sql("""
    CREATE OR REPLACE TABLE raw_spend AS
    SELECT * FROM read_csv('../data/raw/*.csv', all_varchar = true, filename = true)
""")

CountBIGINT
77033


## 2. Build a cleaned table
Converts amounts and dates to proper types, trims text, labels blank service areas as Unknown and flags redacted (personal data) payments.

In [2]:
con.sql("""
    CREATE OR REPLACE TABLE clean_spend AS
    SELECT
        regexp_extract(filename, '(january|february|march|april|may|june)') AS file_month,
        COALESCE(NULLIF(trim("Service Area"), ''), 'Unknown')               AS service_area,
        trim("Expense Type")                                                AS expense_type,
        trim("Description")                                                 AS description,
        trim("SAP Document Number")                                         AS sap_doc_number,
        CAST(try_strptime(trim("Posting Date"), '%d/%m/%Y %H:%M') AS DATE)  AS posting_date,
        trim("Vendor")                                                      AS vendor_raw,
        TRY_CAST(replace(replace(trim("Actual Value"), '£', ''), ',', '') AS DECIMAL(14,2)) AS amount,
        trim("Vendor") ILIKE '%REDACTED%'                                   AS is_redacted
    FROM raw_spend
""")
con.sql("SELECT * FROM clean_spend LIMIT 5").show()

┌────────────┬───────────────────────────────┬───────────────────────────┬────────────────────────────┬────────────────┬──────────────┬─────────────────────────────────────┬───────────────┬─────────────┐
│ file_month │         service_area          │       expense_type        │        description         │ sap_doc_number │ posting_date │             vendor_raw              │    amount     │ is_redacted │
│  varchar   │            varchar            │          varchar          │          varchar           │    varchar     │     date     │               varchar               │ decimal(14,2) │   boolean   │
├────────────┼───────────────────────────────┼───────────────────────────┼────────────────────────────┼────────────────┼──────────────┼─────────────────────────────────────┼───────────────┼─────────────┤
│ april      │ Capital Regen                 │ Miscellaneous - General   │ EXPENDITURE                │ 5101102971     │ 2024-04-20   │ FIVE PAPER                          │       4700

## 3. Data quality checks
Nine checks: row counts, blanks, duplicates, conversion failures, refunds, odd dates, redacted payments, truncated names and supplier spelling variants.

In [3]:
print("1. Rows per file")
con.sql("""
    SELECT regexp_extract(filename, '(january|february|march|april|may|june)') AS file_month,
           COUNT(*) AS row_count
    FROM raw_spend
    GROUP BY 1
    ORDER BY row_count DESC
""").show()

print("2. Blank values per column")
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE "Service Area" IS NULL OR trim("Service Area") = '')               AS service_area,
        COUNT(*) FILTER (WHERE "Expense Type" IS NULL OR trim("Expense Type") = '')               AS expense_type,
        COUNT(*) FILTER (WHERE "Description" IS NULL OR trim("Description") = '')                 AS description,
        COUNT(*) FILTER (WHERE "SAP Document Number" IS NULL OR trim("SAP Document Number") = '') AS sap_doc,
        COUNT(*) FILTER (WHERE "Posting Date" IS NULL OR trim("Posting Date") = '')               AS posting_date,
        COUNT(*) FILTER (WHERE "Vendor" IS NULL OR trim("Vendor") = '')                           AS vendor,
        COUNT(*) FILTER (WHERE "Actual Value" IS NULL OR trim("Actual Value") = '')               AS amount
    FROM raw_spend
""").show()

print("3. Duplicates")
con.sql("""
    SELECT
        (SELECT COUNT(*) FROM raw_spend)
          - (SELECT COUNT(*) FROM (SELECT DISTINCT * EXCLUDE (filename) FROM raw_spend)) AS exact_duplicate_rows,
        (SELECT COUNT(*) FROM (SELECT "SAP Document Number" FROM raw_spend
                               GROUP BY 1 HAVING COUNT(*) > 1))                          AS doc_numbers_used_more_than_once
""").show()

print("4. Did amounts and dates convert?")
con.sql("""
    SELECT COUNT(*) FILTER (WHERE amount IS NULL)       AS amounts_failed,
           COUNT(*) FILTER (WHERE posting_date IS NULL) AS dates_failed
    FROM clean_spend
""").show()

print("5. Negative amounts (refunds)")
con.sql("""
    SELECT COUNT(*) FILTER (WHERE amount < 0)      AS negative_rows,
           SUM(amount) FILTER (WHERE amount < 0)   AS negative_total,
           SUM(amount)                             AS net_total,
           MAX(amount)                             AS largest_payment
    FROM clean_spend
""").show()

print("6. Odd dates")
con.sql("""
    SELECT MIN(posting_date) AS earliest,
           MAX(posting_date) AS latest,
           COUNT(*) FILTER (WHERE posting_date < DATE '2025-01-01')                    AS before_2025,
           COUNT(*) FILTER (WHERE lower(strftime(posting_date, '%B')) <> file_month)   AS date_not_in_file_month
    FROM clean_spend
""").show()

print("7. Redacted (personal data) payments")
con.sql("""
    SELECT COUNT(*)    AS redacted_rows,
           SUM(amount) AS redacted_spend,
           ROUND(100.0 * SUM(amount) / (SELECT SUM(amount) FROM clean_spend), 1) AS pct_of_spend
    FROM clean_spend
    WHERE is_redacted
""").show()

print("8. Supplier names: count and truncation")
con.sql("""
    SELECT COUNT(DISTINCT vendor_raw)  AS distinct_suppliers,
           MAX(length(vendor_raw))     AS longest_name,
           COUNT(DISTINCT vendor_raw) FILTER (
               WHERE length(vendor_raw) = (SELECT MAX(length(vendor_raw)) FROM clean_spend)
           )                           AS names_cut_off_at_max
    FROM clean_spend
""").show()

print("9. Same supplier, different spellings (top 10)")
con.sql("""
    SELECT upper(regexp_replace(vendor_raw, '[^A-Za-z0-9]', '', 'g')) AS match_key,
           COUNT(DISTINCT vendor_raw)                                AS spellings,
           string_agg(DISTINCT vendor_raw, ' | ')                     AS examples
    FROM clean_spend
    WHERE NOT is_redacted
    GROUP BY 1
    HAVING COUNT(DISTINCT vendor_raw) > 1
    ORDER BY spellings DESC
    LIMIT 10
""").show(max_width=200)

1. Rows per file
┌────────────┬───────────┐
│ file_month │ row_count │
│  varchar   │   int64   │
├────────────┼───────────┤
│ may        │     13754 │
│ march      │     13210 │
│ june       │     12933 │
│ january    │     12808 │
│ april      │     12710 │
│ february   │     11618 │
└────────────┴───────────┘

2. Blank values per column
┌──────────────┬──────────────┬─────────────┬─────────┬──────────────┬────────┬────────┐
│ service_area │ expense_type │ description │ sap_doc │ posting_date │ vendor │ amount │
│    int64     │    int64     │    int64    │  int64  │    int64     │ int64  │ int64  │
├──────────────┼──────────────┼─────────────┼─────────┼──────────────┼────────┼────────┤
│            2 │            0 │           0 │       0 │            0 │      0 │      0 │
└──────────────┴──────────────┴─────────────┴─────────┴──────────────┴────────┴────────┘

3. Duplicates
┌──────────────────────┬─────────────────────────────────┐
│ exact_duplicate_rows │ doc_numbers_used_more_tha

## 4. Write the data quality report
Saves `data_quality_report.md` in the project folder, filled in with the numbers from the checks above.

In [4]:
def one(sql):
    return con.execute(sql).fetchall()[0]

total = one("SELECT COUNT(*) FROM raw_spend")[0]
blank_sa = one("""SELECT COUNT(*) FROM raw_spend WHERE "Service Area" IS NULL OR trim("Service Area") = ''""")[0]
dup_rows, dup_docs = one("""
    SELECT (SELECT COUNT(*) FROM raw_spend)
             - (SELECT COUNT(*) FROM (SELECT DISTINCT * EXCLUDE (filename) FROM raw_spend)),
           (SELECT COUNT(*) FROM (SELECT "SAP Document Number" FROM raw_spend GROUP BY 1 HAVING COUNT(*) > 1))
""")
neg_rows, neg_total, net_total, largest = one("""
    SELECT COUNT(*) FILTER (WHERE amount < 0), SUM(amount) FILTER (WHERE amount < 0), SUM(amount), MAX(amount)
    FROM clean_spend
""")
earliest, latest, before_2025, off_month = one("""
    SELECT MIN(posting_date), MAX(posting_date),
           COUNT(*) FILTER (WHERE posting_date < DATE '2025-01-01'),
           COUNT(*) FILTER (WHERE lower(strftime(posting_date, '%B')) <> file_month)
    FROM clean_spend
""")
red_rows, red_spend = one("SELECT COUNT(*), SUM(amount) FROM clean_spend WHERE is_redacted")
n_suppliers, max_len, at_max = one("""
    SELECT COUNT(DISTINCT vendor_raw), MAX(length(vendor_raw)),
           COUNT(DISTINCT vendor_raw) FILTER (WHERE length(vendor_raw) = (SELECT MAX(length(vendor_raw)) FROM clean_spend))
    FROM clean_spend
""")
variants = one("""
    SELECT COUNT(*) FROM (
        SELECT upper(regexp_replace(vendor_raw, '[^A-Za-z0-9]', '', 'g'))
        FROM clean_spend WHERE NOT is_redacted
        GROUP BY 1 HAVING COUNT(DISTINCT vendor_raw) > 1)
""")[0]

pct = lambda n: f"{100 * n / total:.1f}%"

report = f"""# Data Quality Report: Liverpool City Council spend over £500 (Jan–Jun 2025)

**Source:** Liverpool City Council, published on Data Mill North. Six monthly CSV files (only January–June 2025 were available).
**Rows loaded:** {total:,}

| # | Check | Finding | Impact / action |
|---|---|---|---|
| 1 | Amount format | Stored as text with £ sign and commas (e.g. "£15,499.00") | Converted to a number; every row converted successfully |
| 2 | Missing values | {blank_sa} rows with a blank Service Area; no other blanks | Kept, labelled "Unknown" |
| 3 | Exact duplicate rows | {dup_rows} rows identical to another row | Could be genuine repeat payments, so kept but noted |
| 4 | Repeated document numbers | {dup_docs:,} SAP document numbers appear more than once | Probably one invoice split across several lines; kept |
| 5 | Negative amounts | {neg_rows:,} rows ({pct(neg_rows)}), totalling -£{abs(neg_total):,.2f} | Refunds or corrections; kept so totals are net |
| 6 | Dates outside the file's month | {off_month:,} rows ({pct(off_month)}) have a posting date outside the month of their file; {before_2025:,} are before 2025 (earliest {earliest}) | Files seem to group payments by when they were reported, not posted. Analysis uses posting date |
| 7 | Redacted suppliers | {red_rows:,} rows ({pct(red_rows)}) show "REDACTED PERSONAL DATA", worth £{red_spend:,.2f} | Payments to individuals; excluded from supplier analysis (GDPR) |
| 8 | Truncated supplier names | Names are capped at {max_len} characters; {at_max} distinct names hit the cap (e.g. "LIVERPOOL STREETSCENE SERVICES LIMI") | Handled in supplier standardisation (Phases 3–4) |
| 9 | Supplier spelling variants | {n_suppliers:,} distinct supplier names; {variants} groups differ only by case, spacing or punctuation (e.g. MERSEYTRAVEL / MerseyTravel / Merseytravel) | Main target for cleaning in Phases 3–4 |

**Net spend in the data:** £{net_total:,.2f}. **Largest single payment:** £{largest:,.2f}.
"""

with open("../data_quality_report.md", "w") as f:
    f.write(report)
print(report)

# Data Quality Report: Liverpool City Council spend over £500 (Jan–Jun 2025)

**Source:** Liverpool City Council, published on Data Mill North. Six monthly CSV files (only January–June 2025 were available).
**Rows loaded:** 77,033

| # | Check | Finding | Impact / action |
|---|---|---|---|
| 1 | Amount format | Stored as text with £ sign and commas (e.g. "£15,499.00") | Converted to a number; every row converted successfully |
| 2 | Missing values | 2 rows with a blank Service Area; no other blanks | Kept, labelled "Unknown" |
| 3 | Exact duplicate rows | 3 rows identical to another row | Could be genuine repeat payments, so kept but noted |
| 4 | Repeated document numbers | 46 SAP document numbers appear more than once | Probably one invoice split across several lines; kept |
| 5 | Negative amounts | 1,221 rows (1.6%), totalling -£15,829,129.42 | Refunds or corrections; kept so totals are net |
| 6 | Dates outside the file's month | 18,847 rows (24.5%) have a posting date outside the